# Módulo 1 — Funções

Funções permitem **reutilizar lógica**, organizar o código e torná-lo testável. Neste notebook construímos funções reais para o setor elétrico.

## O que vamos ver

1. Definição de funções, parâmetros e retorno
2. Funções para cálculo de indicadores (DEC, FEC)
3. Validação de CPF e CNPJ
4. Cálculo de tarifa
5. Lambda functions
6. `*args` e `**kwargs`

---

> **Princípio:** Uma boa função faz **uma** coisa, faz bem e tem nome descritivo. Se você copiar e colar código, é hora de criar uma função.

## 1. Definição Básica de Funções

In [ ]:
# Estrutura básica de uma função
def saudacao(nome: str) -> str:
    """Retorna uma saudação personalizada."""
    return f"Olá, {nome}!"

# Chamando a função
mensagem = saudacao("Analista de Dados")
print(mensagem)

# Função com parâmetro padrão
def formatar_consumo(kwh: float, casas_decimais: int = 1) -> str:
    """Formata um valor de consumo em kWh."""
    return f"{kwh:,.{casas_decimais}f} kWh"

print(formatar_consumo(1234.567))
print(formatar_consumo(1234.567, casas_decimais=3))
print(formatar_consumo(85000))

## 2. Cálculo dos Indicadores DEC e FEC

**DEC** (Duração Equivalente de Interrupção por Consumidor) e **FEC** (Frequência Equivalente) são os principais indicadores de qualidade de fornecimento regulados pela ANEEL.

In [ ]:
from typing import List, Dict

def calcular_dec(
    interrupcoes: List[Dict],
    total_consumidores: int
) -> float:
    """
    Calcula o DEC (Duração Equivalente de Interrupção por Consumidor).
    
    Fórmula:
        DEC = Σ(Di × Ci) / Ct
        onde:
          Di = duração da interrupção i (horas)
          Ci = número de consumidores afetados na interrupção i
          Ct = total de consumidores do conjunto
    
    Args:
        interrupcoes: lista de dicts com 'duracao_horas' e 'consumidores_afetados'
        total_consumidores: total de UCs do conjunto de distribuição
    
    Returns:
        DEC em horas (float)
    
    Raises:
        ValueError: se total_consumidores for zero ou negativo
    """
    if total_consumidores <= 0:
        raise ValueError("total_consumidores deve ser maior que zero")
    
    soma_ponderada = sum(
        i["duracao_horas"] * i["consumidores_afetados"]
        for i in interrupcoes
    )
    
    return round(soma_ponderada / total_consumidores, 4)


def calcular_fec(
    interrupcoes: List[Dict],
    total_consumidores: int
) -> float:
    """
    Calcula o FEC (Frequência Equivalente de Interrupção por Consumidor).
    
    Fórmula:
        FEC = Σ(Ci) / Ct
    
    Args:
        interrupcoes: lista de dicts com 'consumidores_afetados'
        total_consumidores: total de UCs do conjunto
    
    Returns:
        FEC em número de interrupções por consumidor (float)
    """
    if total_consumidores <= 0:
        raise ValueError("total_consumidores deve ser maior que zero")
    
    soma_afetados = sum(i["consumidores_afetados"] for i in interrupcoes)
    return round(soma_afetados / total_consumidores, 4)


# Exemplo: conjunto de distribuição com 10.000 UCs
total_ucs = 10000

interrupcoes_janeiro = [
    {"id": "INT-001", "data": "05/01/2025", "duracao_horas": 2.5,  "consumidores_afetados": 1500},
    {"id": "INT-002", "data": "12/01/2025", "duracao_horas": 0.75, "consumidores_afetados": 300},
    {"id": "INT-003", "data": "18/01/2025", "duracao_horas": 6.0,  "consumidores_afetados": 4200},
    {"id": "INT-004", "data": "25/01/2025", "duracao_horas": 1.25, "consumidores_afetados": 800},
]

dec_jan = calcular_dec(interrupcoes_janeiro, total_ucs)
fec_jan = calcular_fec(interrupcoes_janeiro, total_ucs)

print(f"=== Indicadores de Janeiro/2025 ===")
print(f"Total de UCs: {total_ucs:,}")
print(f"Número de interrupções: {len(interrupcoes_janeiro)}")
print(f"DEC: {dec_jan:.4f} horas/consumidor")
print(f"FEC: {fec_jan:.4f} interrupções/consumidor")

# Comparar com limites (fictícios para exemplo)
limite_dec = 15.0  # horas/ano
limite_fec = 12.0  # interrupções/ano
dec_acumulado_estimado = dec_jan * 12  # estimativa anual

print(f"\nEstimativa anual: DEC = {dec_acumulado_estimado:.2f} h (limite: {limite_dec} h)")
situacao = "CONFORME" if dec_acumulado_estimado <= limite_dec else "ACIMA DO LIMITE"
print(f"Situação: {situacao}")

## 3. Validação de CPF e CNPJ

In [ ]:
import re

def limpar_documento(documento: str) -> str:
    """Remove caracteres não numéricos de um CPF ou CNPJ."""
    return re.sub(r'\D', '', str(documento))


def validar_cpf(cpf: str) -> bool:
    """
    Valida um CPF verificando formato e dígitos verificadores.
    
    Args:
        cpf: string com CPF (com ou sem formatação)
    
    Returns:
        True se o CPF é válido, False caso contrário
    """
    cpf = limpar_documento(cpf)
    
    # Verificações básicas
    if len(cpf) != 11:
        return False
    if cpf == cpf[0] * 11:  # CPFs com todos os dígitos iguais são inválidos
        return False
    
    # Primeiro dígito verificador
    soma = sum(int(cpf[i]) * (10 - i) for i in range(9))
    resto = soma % 11
    digito1 = 0 if resto < 2 else 11 - resto
    
    if int(cpf[9]) != digito1:
        return False
    
    # Segundo dígito verificador
    soma = sum(int(cpf[i]) * (11 - i) for i in range(10))
    resto = soma % 11
    digito2 = 0 if resto < 2 else 11 - resto
    
    return int(cpf[10]) == digito2


def formatar_cpf(cpf: str) -> str:
    """Formata CPF para o padrão XXX.XXX.XXX-XX."""
    c = limpar_documento(cpf)
    if len(c) != 11:
        return cpf  # retorna original se inválido
    return f"{c[:3]}.{c[3:6]}.{c[6:9]}-{c[9:]}"


def validar_cnpj(cnpj: str) -> bool:
    """
    Valida um CNPJ verificando formato e dígitos verificadores.
    """
    cnpj = limpar_documento(cnpj)
    
    if len(cnpj) != 14:
        return False
    if cnpj == cnpj[0] * 14:
        return False
    
    # Pesos para cálculo do primeiro dígito
    pesos1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    soma = sum(int(cnpj[i]) * pesos1[i] for i in range(12))
    resto = soma % 11
    digito1 = 0 if resto < 2 else 11 - resto
    
    if int(cnpj[12]) != digito1:
        return False
    
    # Pesos para o segundo dígito
    pesos2 = [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    soma = sum(int(cnpj[i]) * pesos2[i] for i in range(13))
    resto = soma % 11
    digito2 = 0 if resto < 2 else 11 - resto
    
    return int(cnpj[13]) == digito2


def identificar_tipo_documento(documento: str) -> str:
    """Identifica se o documento é CPF (11 dígitos) ou CNPJ (14 dígitos)."""
    limpo = limpar_documento(documento)
    if len(limpo) == 11:
        return "CPF"
    elif len(limpo) == 14:
        return "CNPJ"
    return "DESCONHECIDO"


# Testes
documentos_teste = [
    "529.982.247-25",    # CPF válido
    "111.111.111-11",    # CPF inválido (todos iguais)
    "123.456.789-00",    # CPF inválido
    "11.222.333/0001-81",# CNPJ válido
    "00.000.000/0000-00",# CNPJ inválido
]

print(f"{'Documento':<25} | {'Tipo':<8} | {'Válido':<8}")
print("-" * 48)
for doc in documentos_teste:
    tipo = identificar_tipo_documento(doc)
    if tipo == "CPF":
        valido = validar_cpf(doc)
    elif tipo == "CNPJ":
        valido = validar_cnpj(doc)
    else:
        valido = False
    
    status = "Sim" if valido else "Não"
    print(f"{doc:<25} | {tipo:<8} | {status:<8}")

## 4. Cálculo de Tarifa

In [ ]:
# Tabela de tarifas (valores fictícios para fins educacionais)
TARIFAS = {
    "B1": {"te": 0.4521, "tusd": 0.3991, "descricao": "Residencial"},
    "B2": {"te": 0.3845, "tusd": 0.2987, "descricao": "Rural"},
    "B3": {"te": 0.4120, "tusd": 0.3234, "descricao": "Comercial"},
    "A4": {"te": 0.3210, "tusd": 0.2156, "descricao": "Média Tensão"},
}


def calcular_valor_fatura(
    consumo_kwh: float,
    modalidade: str,
    desconto_baixa_renda: float = 0.0,
    tributos_pct: float = 0.3258  # ICMS + PIS/COFINS médio
) -> dict:
    """
    Calcula o valor estimado da fatura de energia.
    
    Args:
        consumo_kwh: consumo mensal em kWh
        modalidade: modalidade tarifária (ex: 'B1', 'B3', 'A4')
        desconto_baixa_renda: percentual de desconto (0 a 1, ex: 0.40 = 40%)
        tributos_pct: alíquota total de tributos
    
    Returns:
        dict com componentes da fatura
    """
    if modalidade not in TARIFAS:
        raise ValueError(f"Modalidade '{modalidade}' não encontrada. Use: {list(TARIFAS.keys())}")
    
    tarifa = TARIFAS[modalidade]
    tarifa_total_kwh = tarifa["te"] + tarifa["tusd"]
    
    # Valor base sem tributos
    valor_base = consumo_kwh * tarifa_total_kwh
    
    # Aplicar desconto baixa renda
    valor_apos_desconto = valor_base * (1 - desconto_baixa_renda)
    
    # Tributos
    valor_tributos = valor_apos_desconto * tributos_pct
    
    # Total
    valor_total = valor_apos_desconto + valor_tributos
    
    return {
        "consumo_kwh": consumo_kwh,
        "modalidade": modalidade,
        "descricao_modalidade": tarifa["descricao"],
        "tarifa_te_kwh": tarifa["te"],
        "tarifa_tusd_kwh": tarifa["tusd"],
        "tarifa_total_kwh": tarifa_total_kwh,
        "valor_base_rs": round(valor_base, 2),
        "desconto_rs": round(valor_base - valor_apos_desconto, 2),
        "tributos_rs": round(valor_tributos, 2),
        "valor_total_rs": round(valor_total, 2)
    }


# Exemplos de cálculo
exemplos = [
    {"consumo": 250, "modalidade": "B1", "desconto": 0.0},
    {"consumo": 80,  "modalidade": "B1", "desconto": 0.40},  # Baixa Renda
    {"consumo": 3500, "modalidade": "B3", "desconto": 0.0},
]

for ex in exemplos:
    resultado = calcular_valor_fatura(ex["consumo"], ex["modalidade"], ex["desconto"])
    print(f"\n--- {resultado['descricao_modalidade']} ({resultado['modalidade']}) ---")
    print(f"  Consumo: {resultado['consumo_kwh']} kWh")
    print(f"  Tarifa total: R$ {resultado['tarifa_total_kwh']:.4f}/kWh")
    print(f"  Valor base: R$ {resultado['valor_base_rs']:.2f}")
    if resultado['desconto_rs'] > 0:
        print(f"  Desconto B. Renda: -R$ {resultado['desconto_rs']:.2f}")
    print(f"  Tributos: R$ {resultado['tributos_rs']:.2f}")
    print(f"  TOTAL: R$ {resultado['valor_total_rs']:.2f}")

## 5. Lambda Functions

Lambdas são funções anônimas de uma linha — úteis como argumentos para `sorted()`, `map()`, `filter()`.

In [ ]:
# Lambda simples
kwh_para_mwh = lambda kwh: kwh / 1000
print(f"85.000 kWh = {kwh_para_mwh(85000)} MWh")

# Lambda com múltiplos argumentos
calcular_demanda_reativa = lambda p_kw, fp: p_kw * (1 - fp**2)**0.5 / fp if fp > 0 else 0

print(f"Demanda reativa (1000 kW, fp=0.92): {calcular_demanda_reativa(1000, 0.92):.1f} kVAr")

# Ordenação com lambda
consumidores = [
    {"nome": "João", "consumo": 245.0, "classe": "RESIDENCIAL"},
    {"nome": "Empresa A", "consumo": 8920.5, "classe": "COMERCIAL"},
    {"nome": "Ana", "consumo": 89.3, "classe": "RESIDENCIAL"},
    {"nome": "Fábrica B", "consumo": 85000.0, "classe": "INDUSTRIAL"},
]

# Ordenar por consumo (crescente)
por_consumo = sorted(consumidores, key=lambda c: c["consumo"])
print("\nOrdenado por consumo (crescente):")
for c in por_consumo:
    print(f"  {c['nome']}: {c['consumo']:,.1f} kWh")

# Filtrar com lambda usando filter()
alta_tensao = list(filter(lambda c: c["consumo"] > 1000, consumidores))
print(f"\nConsumidores com consumo > 1.000 kWh: {len(alta_tensao)}")

# Transformar com map()
consumos_mwh = list(map(lambda c: {"nome": c["nome"], "consumo_mwh": c["consumo"] / 1000}, consumidores))
print("\nConsumo em MWh:")
for c in consumos_mwh:
    print(f"  {c['nome']}: {c['consumo_mwh']:.3f} MWh")

## 6. `*args` e `**kwargs`

`*args` recebe qualquer número de argumentos posicionais. `**kwargs` recebe argumentos nomeados. São essenciais para funções genéricas e flexíveis.

In [ ]:
# *args — número variável de leituras
def calcular_media_leituras(*leituras: float) -> dict:
    """
    Calcula estatísticas de uma série de leituras.
    Aceita qualquer número de valores.
    """
    if not leituras:
        return {"erro": "Nenhuma leitura fornecida"}
    
    validas = [l for l in leituras if l is not None and l >= 0]
    
    return {
        "n_leituras": len(validas),
        "media": round(sum(validas) / len(validas), 2),
        "minimo": min(validas),
        "maximo": max(validas),
        "amplitude": max(validas) - min(validas)
    }

stats = calcular_media_leituras(310, 325, 342, 298, 315, 401, 389)
print("Estatísticas das leituras:")
for chave, valor in stats.items():
    print(f"  {chave}: {valor}")

# **kwargs — metadados de qualidade
def registrar_problema_qualidade(
    cod_consumidor: int,
    tipo_problema: str,
    **detalhes
) -> dict:
    """
    Registra um problema de qualidade de cadastro.
    **detalhes aceita qualquer informação adicional.
    """
    registro = {
        "cod_consumidor": cod_consumidor,
        "tipo_problema": tipo_problema,
        "detalhes": detalhes
    }
    return registro

# Usando com diferentes tipos de problema
p1 = registrar_problema_qualidade(
    100042, "CPF_INVALIDO",
    cpf_fornecido="000.000.000-00",
    campo="cpf_cnpj"
)

p2 = registrar_problema_qualidade(
    100043, "CONSUMO_OUTLIER",
    consumo_registrado=9999999,
    media_historica=8920.5,
    desvios_acima=45.2
)

print("\nProblemas registrados:")
for prob in [p1, p2]:
    print(f"  UC {prob['cod_consumidor']}: {prob['tipo_problema']}")
    for k, v in prob['detalhes'].items():
        print(f"    {k}: {v}")